<img src="randle-1.png" width="300" style="margin-left: 100px;" style="margin-right: 100px;" /><img src="randle-2.png" width="310" style="margin-left: 100px;" style="margin-right: 100px;" />

# Randle Cycle ODE Modelling Notes

## 0. Notation and State Variables

We use reaction rates $v_{\text{X}}$ as kinetic terms, to be defined later.

### Glucose Path – Main State Variables (CYTO + MITO)

- $\text{Glc}_{\text{ext}}(t)$ – extracellular glucose  
- $\text{Glc}(t)$ – cytosolic glucose  
- $\text{Glc-6-P}(t)$ – glucose 6-phosphate  
- $\text{Fru-6-P}(t)$ – fructose 6-phosphate  
- $\text{Fru-1,6-P}_2(t)$ – fructose 1,6-bisphosphate  
- $\text{Fru-2,6-P}_2(t)$ – fructose 2,6-bisphosphate  
- $\text{Pyruvate}(t)$ – pyruvate (cytosol/mito entry pool)  
- $\text{Acetyl-CoA}_{\text{Glc}}(t)$ – acetyl-CoA from glucose  
- $\text{Cit}(t)$ – citrate pool from Acetyl-CoA    
- $\text{Glycogen}(t)$ – glycogen store  

### Fat Path – Main State Variables (CYTO + MITO)

- $\text{LCFA}_{\text{ext}}(t)$ – extracellular long-chain fatty acids  
- $\text{LCFA-CoA}_{\text{cyto}}(t)$ – cytosolic LCFA-CoA  
- $\text{LCFA-CoA}_{\text{mito}}(t)$ – mitochondrial LCFA-CoA  
- $\text{Acetyl-CoA}_{\text{Fat}}(t)$ – acetyl-CoA from fat  
- $\text{Mal}(t)$ – malonyl-CoA  
- $\text{TAG}(t)$ – triglyceride store  

We also have $\text{Glc}_{\text{in}}(t)$, $\text{LCFA}_{\text{in}}(t)$, $\text{ATP}_{\text{draw}}(t)$ as user inputs.

---

## 1. Glucose Path ODEs

### 1.1 Glucose Uptake and Phosphorylation

Extracellular to cytosolic glucose via GLUT4:

$$
\frac{d\,\text{Glc}_{\text{ext}}}{dt}
= \text{Glc}_{\text{in}} - v_{\text{GLUT4}}
$$

where $\text{Glc}_{\text{in}}$ is the user-defined glucose input rate.

$$
\frac{d\,\text{Glc}}{dt}
= v_{\text{GLUT4}} - v_{\text{HK}}
$$

Hexokinase (HK) converts Glc to Glc-6-P:

$$
\frac{d\,\text{Glc-6-P}}{dt}
= v_{\text{HK}} - v_{\text{PGI}} - v_{\text{Glycogen synthase}}
$$

### 1.2 PFK and Glycolytic Intermediates

Isomerization to Fru-6-P:

$$
\frac{d\,\text{Fru-6-P}}{dt}
= v_{\text{PGI}} - v_{\text{PFK}}
$$

PFK produces Fru-1,6-P$_2$:

$$
\frac{d\,\text{Fru-1,6-P}_2}{dt}
= v_{\text{PFK}} - v_{\text{LG}}
$$

Downstream glycolytic flux (lumping ALD + downstream steps) to pyruvate:

$$
\frac{d\,\text{Pyruvate}}{dt}
= 2*v_{\text{LG}} - v_{\text{PDH}} - v_{\text{LDH}}
$$

LDH is step from Pyruvate -> Lactate

### 1.3 Fru-2,6-P$_2$ Regulation

PFK-2 produces Fru-2,6-P$_2$:

$$
\frac{d\,\text{Fru-2,6-P}_2}{dt}
= v_{\text{PFK2,prod}} - v_{\text{PFK2,deg}}
$$

Fru-2,6-P$_2$ will appear as an **activator** in $v_{\text{PFK}}$.

In this reduced Randle-cycle model, we do not track Fru-2,6-P$_2$ or PFK-2 explicitly.

Their effects on PFK activity are assumed to be absorbed into the effective
parameters of $v_{\text{PFK}}$ (e.g., $V_{\max,\text{PFK}}$, $K_{m,\text{PFK}}$).

The dominant regulatory signal we retain is **citrate inhibition of PFK**,
linking mitochondrial fat oxidation to suppression of glycolysis.

### 1.4 PDH and Citrate

Pyruvate to acetyl-CoA via PDH:

$$
\frac{d\,\text{Acetyl-CoA}_{\text{Glu}}}{dt}
= v_{\text{PDH}} - \text{ATP}_{\text{draw}}
$$

#### Citrate – Single Pool ODE

Let $\text{Cit}(t)$ be the effective citrate pool (signal).

$$
\frac{d\,\text{Cit}}{dt}
= k_{\text{Cit,prod}} \cdot \text{AcCoA}_{\text{Glu}}
- k_{\text{Cit,clear}} \cdot \text{Cit}
$$

- $k_{\text{Cit,prod}}$: effective citrate production rate from glucose-derived acetyl-CoA  
- $k_{\text{Cit,clear}}$: lumped citrate clearance (ACL + TCA + other sinks)


### 1.5 Glycogen Synthesis (optional)

From Glc-6-P:

$$
\frac{d\,\text{Glycogen}}{dt}
= v_{\text{Glycogen synthase}} - v_{\text{Glycogen breakdown}}
$$

---

## 2. Fat Path ODEs

### 2.1 LCFA Uptake and Activation

Extracellular LCFA via CD36:

$$
\frac{d\,\text{LCFA}_{\text{ext}}}{dt}
= \text{LCFA}_{\text{in}} - v_{\text{CD36}}
$$

Activation to LCFA-CoA in cytosol:

$$
\frac{d\,\text{LCFA-CoA}_{\text{cyto}}}{dt}
= v_{\text{CD36}} - v_{\text{CPT1}} - v_{\text{TAG synthase}}
$$

### 2.2 CPT1 Transport and β-Oxidation

Transport into mitochondria:

$$
\frac{d\,\text{LCFA-CoA}_{\text{mito}}}{dt}
= v_{\text{CPT1}} - v_{\beta\text{-ox}}
$$

β-oxidation to acetyl-CoA:

$$
\frac{d\,\text{Acetyl-CoA}_{\text{Fat}}}{dt}
= v_{\beta\text{-ox}} - v_{\text{TCA,Fat}} - v_{\text{ACC substrate,Fat}}
$$

### 2.3 Malonyl-CoA and TAG

Malonyl-CoA from ACC (using acetyl-CoA and/or citrate):

$$
\frac{d\,\text{Mal}}{dt}
= v_{\text{ACC}}(\text{Acetyl-CoA}_{\text{Glu}}, \text{Cit}_{\text{cyto}})
- v_{\text{MCD}}(\text{Mal})
$$

TAG synthesis from LCFA-CoA:

$$
\frac{d\,\text{TAG}}{dt}
= v_{\text{TAG synthase}}(\text{LCFA-CoA}_{\text{cyto}})
- v_{\text{TAG breakdown}}
$$

---

These notes give you a **structured, intermediate‑level ODE system** for both glucose and fat paths.
We can later refine each $v_{\text{X}}$ into explicit kinetic rate laws (Michaelis–Menten, reversible, Hill, etc.) and start assigning parameter values.


# Kinetic Terms – $v_{GLUT4}$

We define the GLUT4 uptake flux with citrate inhibition:

## Full Expression

$$
v_{\text{GLUT4}}
= V_{\max,\text{GLUT4}}
\cdot \frac{\text{Glc}_{\text{ext}}}{K_{m,\text{GLUT4}} + \text{Glc}_{\text{ext}}}
\cdot \frac{1}{1 + \alpha_{\text{Cit,GLUT4}}\,\text{Cit}}
$$


<ul>
  <li><strong>$V_{\max,\text{GLUT4}}$</strong> — maximal GLUT4 transport capacity
    <ul>
      <li>Sets the upper limit of glucose uptake.</li>
      <li>Units: concentration/time.</li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{GLUT4}}$</strong> — Michaelis constant
    <ul>
      <li>External glucose level at half-maximal uptake.</li>
      <li>Controls sensitivity to $\text{Glc}_{\text{ext}}$.</li>
    </ul>
  </li>

  <li><strong>$\alpha_{\text{Cit,GLUT4}}$</strong> — citrate inhibition coefficient
    <ul>
      <li>Strength of citrate suppression on GLUT4 flux.</li>
      <li>Units: inverse concentration of citrate.</li>
    </ul>
  </li>
</ul>

## Example Parameter Values and Tuning Ranges

<ul>
  <li><strong>$V_{\max,\text{GLUT4}}$</strong>
    <ul>
      <li>Example value: <strong>1.0</strong> (model concentration/time units)</li>
      <li>Tuning range: <strong>0.5–5</strong></li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{GLUT4}}$</strong>
    <ul>
      <li>Example value: <strong>5 mM</strong></li>
      <li>Tuning range: <strong>3–7 mM</strong></li>
    </ul>
  </li>

  <li><strong>$\alpha_{\text{Cit,GLUT4}}$</strong>
    <ul>
      <li>Example value: <strong>2 1/mM</strong></li>
      <li>Tuning range: <strong>0.5–5 1/mM</strong></li>
    </ul>
  </li>
</ul>


# Kinetic Terms – $v_{HK}$

We now define the **hexokinase flux** converting cytosolic glucose to Glc‑6‑P.

## Full Expression

$$
v_{\text{HK}}
= V_{\max,\text{HK}}
\cdot \frac{\text{Glc}}{K_{m,\text{HK}} + \text{Glc}}
\cdot \frac{1}{1 + \alpha_{\text{Glc6P,HK}} \,\text{Glc-6-P}}
$$


- $V_{\max,\text{HK}}$ – maximal HK rate (concentration/time)  
- $K_{m,\text{HK}}$ – HK Michaelis constant for glucose (concentration)  
- $\alpha_{\text{Glc6P,HK}}$ – strength of Glc‑6‑P product inhibition  
  - Units: inverse concentration of Glc‑6‑P

## Example Parameter Values and Tuning Ranges

<ul>
  <li><strong>$V_{\max,\text{HK}}$</strong>
    <ul>
      <li>Example value: <strong>1.0</strong> (model concentration/time units)</li>
      <li>Tuning range: <strong>0.5–5</strong></li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{HK}}$</strong>
    <ul>
      <li>Example value: <strong>0.1 mM</strong> (muscle/adipose hexokinase, high affinity)</li>
      <li>Tuning range: <strong>0.03–0.3 mM</strong></li>
    </ul>
  </li>

  <li><strong>$\alpha_{\text{Glc6P,HK}}$</strong>
    <ul>
      <li>Example value: <strong>2 1/mM</strong></li>
      <li>Tuning range: <strong>0.5–5 1/mM</strong></li>
    </ul>
  </li>
</ul>


# Kinetic Terms – $v_{\text{PGI}}$

Phosphoglucose isomerase converts **Glc‑6‑P → F6P**.  
This step is fast and near‑equilibrium, so we use a simple reversible MM‑style term.

## Full Expression

$$
v_{\text{PGI}}
= V_{\max,\text{PGI}}
\cdot 
\left(
\frac{\text{Glc6P}}{K_{m,\text{PGI}} + \text{Glc6P}}
-
\frac{\text{F6P}}{K_{m,\text{PGI}} + \text{F6P}}
\right)
$$

- $V_{\max,\text{PGI}}$ – maximal PGI catalytic rate (concentration/time)  
- $K_{m,\text{PGI}}$ – Michaelis constant for both Glc‑6‑P and F6P  
- No inhibition term (PGI is not strongly regulated in Randle‑cycle dynamics)

## Example Parameter Values and Tuning Ranges

<ul>
  <li><strong>$V_{\max,\text{PGI}}$</strong>
    <ul>
      <li>Example value: <strong>5.0</strong> (model concentration/time units)</li>
      <li>Tuning range: <strong>2–10</strong></li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{PGI}}$</strong>
    <ul>
      <li>Example value: <strong>0.1 mM</strong></li>
      <li>Tuning range: <strong>0.05–0.3 mM</strong></li>
    </ul>
  </li>
</ul>


# Kinetic Terms – $v_{\text{GS}}$

Glycogen Synthase converts **Glc‑6‑P → Glycogen**.  
We model it with activation by Glc‑6‑P (allosteric) and a simple MM dependence on substrate.

## Full Expression

$$
v_{\text{GS}}
= V_{\max,\text{GS}}
\cdot \frac{\text{Glc6P}}{K_{m,\text{GS}} + \text{Glc6P}}
\cdot \left( 1 + \beta_{\text{Glc6P,GS}} \cdot \text{Glc6P} \right)
$$

- $V_{\max,\text{GS}}$ — maximal glycogen synthase rate (concentration/time)  
- $K_{m,\text{GS}}$ — Michaelis constant for Glc‑6‑P  
- $\beta_{\text{Glc6P,GS}}$ — activation strength by Glc‑6‑P  
  - Units: inverse concentration of Glc‑6‑P  
  - Higher $\beta$ → stronger activation

This is a **phenomenological activation term**, capturing the fact that Glc‑6‑P strongly stimulates GS activity.

## Example Parameter Values and Tuning Ranges

<ul>
  <li><strong>$V_{\max,\text{GS}}$</strong>
    <ul>
      <li>Example value: <strong>0.5</strong> (model concentration/time units)</li>
      <li>Tuning range: <strong>0.2–2.0</strong></li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{GS}}$</strong>
    <ul>
      <li>Example value: <strong>0.05 mM</strong></li>
      <li>Tuning range: <strong>0.02–0.2 mM</strong></li>
    </ul>
  </li>

  <li><strong>$\beta_{\text{Glc6P,GS}}$</strong>
    <ul>
      <li>Example value: <strong>1.0 1/mM</strong></li>
      <li>Tuning range: <strong>0.5–3.0 1/mM</strong></li>
    </ul>
  </li>
</ul>


# Kinetic Terms – $v_{\text{PFK}}$

PFK‑1 converts **F6P → F1,6BP** and is inhibited by citrate.

## Full Expression

$$
v_{\text{PFK}}
= V_{\max,\text{PFK}}
\cdot \frac{\text{F6P}}{K_{m,\text{PFK}} + \text{F6P}}
\cdot \frac{1}{1 + \alpha_{\text{Cit,PFK}} \,\text{Cit}}
$$

- $V_{\max,\text{PFK}}$ – maximal PFK rate  
- $K_{m,\text{PFK}}$ – Michaelis constant for F6P  
- $\alpha_{\text{Cit,PFK}}$ – citrate inhibition strength (units: inverse citrate concentration)

## Example Parameter Values and Tuning Ranges

<ul>
  <li><strong>$V_{\max,\text{PFK}}$</strong>
    <ul>
      <li>Example value: <strong>2.0</strong></li>
      <li>Tuning range: <strong>1–10</strong></li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{PFK}}$</strong>
    <ul>
      <li>Example value: <strong>0.1 mM</strong></li>
      <li>Tuning range: <strong>0.05–0.3 mM</strong></li>
    </ul>
  </li>

  <li><strong>$\alpha_{\text{Cit,PFK}}$</strong>
    <ul>
      <li>Example value: <strong>2.0 1/mM</strong></li>
      <li>Tuning range: <strong>0.5–5.0 1/mM</strong></li>
    </ul>
  </li>
</ul>


# Kinetic Terms – Lumped Lower Glycolysis ($v_{\text{LG}}$)

We lump the steps from F1,6BP to pyruvate into a single effective flux.

## Full Expression

$$
v_{\text{LG}}
= V_{\max,\text{LG}}
\cdot \frac{\text{F1,6BP}}{K_{m,\text{LG}} + \text{F1,6BP}}
$$

- $V_{\max,\text{LG}}$ – maximal lower‑glycolysis rate (concentration/time)  
- $K_{m,\text{LG}}$ – effective Michaelis constant for F1,6BP  

## Example Parameter Values and Tuning Ranges

<ul>
  <li><strong>$V_{\max,\text{LG}}$</strong>
    <ul>
      <li>Example value: <strong>5.0</strong></li>
      <li>Tuning range: <strong>2–10</strong></li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{LG}}$</strong>
    <ul>
      <li>Example value: <strong>0.05 mM</strong></li>
      <li>Tuning range: <strong>0.02–0.2 mM</strong></li>
    </ul>
  </li>
</ul>


# Kinetic Terms – $v_{\text{PDH}}$

Pyruvate dehydrogenase (PDH) converts **Pyruvate → Acetyl‑CoA** and is a key control point in the Randle cycle.  
It is inhibited by mitochondrial **acetyl‑CoA** (reflecting fat oxidation).

## Full Expression

$$
v_{\text{PDH}}
= V_{\max,\text{PDH}}
\cdot \frac{\text{Pyruvate}}{K_{m,\text{PDH}} + \text{Pyruvate}}
\cdot \frac{1}{1 + \alpha_{\text{Acetyl-CoA,PDH}} \,\text{Acetyl-CoA}}
$$

- $V_{\max,\text{PDH}}$ – maximal PDH rate (concentration/time)  
- $K_{m,\text{PDH}}$ – Michaelis constant for pyruvate  
- $\alpha_{\text{Acetyl-CoA,PDH}}$ – inhibition strength by acetyl‑CoA (inverse acetyl‑CoA concentration)

## Example Parameter Values and Tuning Ranges

<ul>
  <li><strong>$V_{\max,\text{PDH}}$</strong>
    <ul>
      <li>Example value: <strong>2.0</strong> (model concentration/time units)</li>
      <li>Tuning range: <strong>1–10</strong></li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{PDH}}$</strong>
    <ul>
      <li>Example value: <strong>0.05 mM</strong></li>
      <li>Tuning range: <strong>0.02–0.2 mM</strong></li>
    </ul>
  </li>

  <li><strong>$\alpha_{\text{Acetyl-CoA,PDH}}$</strong>
    <ul>
      <li>Example value: <strong>2.0 1/mM</strong></li>
      <li>Tuning range: <strong>0.5–5.0 1/mM</strong></li>
    </ul>
  </li>
</ul>



# Kinetic Terms – $v_{\text{LDH}}$

Lactate dehydrogenase (LDH) converts **Pyruvate ↔ Lactate**.  
Here we focus on the forward direction (pyruvate → lactate) as a sink for pyruvate.

## Full Expression

$$
v_{\text{LDH}}
= V_{\max,\text{LDH}}
\cdot \frac{\text{Pyruvate}}{K_{m,\text{LDH}} + \text{Pyruvate}}
$$

- $V_{\max,\text{LDH}}$ – maximal LDH rate (concentration/time)  
- $K_{m,\text{LDH}}$ – Michaelis constant for pyruvate  

(If you later add an explicit lactate pool, you can make this reversible.)

## Example Parameter Values and Tuning Ranges

<ul>
  <li><strong>$V_{\max,\text{LDH}}$</strong>
    <ul>
      <li>Example value: <strong>2.0</strong></li>
      <li>Tuning range: <strong>1–10</strong></li>
    </ul>
  </li>

  <li><strong>$K_{m,\text{LDH}}$</strong>
    <ul>
      <li>Example value: <strong>0.05 mM</strong></li>
      <li>Tuning range: <strong>0.02–0.2 mM</strong></li>
    </ul>
  </li>
</ul>
